# CEG 外部 baseline 产物 Colab Notebook

该 Notebook 在独立 Colab 冷启动会话中运行外部 baseline command plan。它可以读取图像生成阶段归档, 但不实现 CEG 主方法, 也不调用 CEG-WM。完成后会把 baseline observations 和 execution manifest 打包回 Google Drive, 供论文结果包阶段读取。


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/RICHAAARC/CEG.git"
REPO_BRANCH = ""
REPO_ROOT = Path("/content/CEG")
DRIVE_ROOT = Path("/content/drive/MyDrive/CEG")
LOCAL_RUNTIME_ROOT = Path("/content/ceg_runtime")

IMAGE_GENERATION_RUN_ID = "paper_main_probe_image_generation_outputs"
RUN_ID = f"{IMAGE_GENERATION_RUN_ID}_external_baselines"
WORKSPACE = LOCAL_RUNTIME_ROOT / RUN_ID
RESET_LOCAL_RUNTIME_WORKSPACE = True
IMAGE_GENERATION_ARCHIVE = DRIVE_ROOT / "archives" / "image_generation_outputs" / f"{IMAGE_GENERATION_RUN_ID}.zip"

# baseline plan 必须由用户或前置脚本提供。该 Notebook 不生成第三方 baseline 方法本身。
BASELINE_PLAN = DRIVE_ROOT / "external_baseline_plans" / "baseline_plan.json"
BASELINE_OUTPUT_ROOT = WORKSPACE / "external_baselines"
RUN_EXTERNAL_BASELINES = True
REQUIRE_BASELINE_PASS = True
BASELINE_FORMAL_RESULT_CLAIM = False
BASELINE_EVIDENCE_PATHS = []


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print(f"非 Colab 环境或 Drive 已挂载: {exc}")

if not REPO_ROOT.exists():
    cmd = ["git", "clone"]
    if REPO_BRANCH:
        cmd += ["--branch", REPO_BRANCH]
    cmd += [REPO_URL, str(REPO_ROOT)]
    print("运行:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "--all", "--prune"], check=True)
    if REPO_BRANCH:
        subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)], check=True)

from paper_workflow.colab_utils.runtime import archive_directory_to_drive, extract_stage_archive

if WORKSPACE.exists() and RESET_LOCAL_RUNTIME_WORKSPACE:
    shutil.rmtree(WORKSPACE)
WORKSPACE.mkdir(parents=True, exist_ok=True)
if IMAGE_GENERATION_ARCHIVE.is_file():
    extract_stage_archive(
        archive_zip_path=IMAGE_GENERATION_ARCHIVE,
        destination_root=WORKSPACE / "inputs" / "images",
        reset=True,
    )
else:
    print("未找到图像生成阶段归档。若 baseline plan 不需要图像输入, 可以继续运行:", IMAGE_GENERATION_ARCHIVE)

print("workspace =", WORKSPACE)
print("baseline_plan =", BASELINE_PLAN, "exists=", BASELINE_PLAN.exists())


In [ ]:
cmd = [
    sys.executable,
    "scripts/run_baseline_plan.py",
    "--plan", str(BASELINE_PLAN),
    "--out", str(BASELINE_OUTPUT_ROOT),
]
if REQUIRE_BASELINE_PASS:
    cmd.append("--require-pass")
if BASELINE_FORMAL_RESULT_CLAIM:
    cmd.append("--formal-result-claim")
for evidence_path in BASELINE_EVIDENCE_PATHS:
    cmd += ["--evidence-path", str(evidence_path)]
print("运行:", " ".join(cmd))
if RUN_EXTERNAL_BASELINES:
    if not BASELINE_PLAN.is_file():
        raise FileNotFoundError(f"baseline plan 不存在: {BASELINE_PLAN}")
    subprocess.run(cmd, check=True)
else:
    print("RUN_EXTERNAL_BASELINES = False, 仅打印命令。")


In [ ]:
archive_manifest = archive_directory_to_drive(
    source_root=BASELINE_OUTPUT_ROOT,
    drive_root=DRIVE_ROOT,
    archive_group="external_baseline_outputs",
    run_id=RUN_ID,
    archive_name=RUN_ID,
)
print("baseline_archive_manifest =")
print(archive_manifest)


In [ ]:
paths = {
    "baseline_plan": BASELINE_PLAN,
    "baseline_observations": BASELINE_OUTPUT_ROOT / "baseline_observations.json",
    "baseline_execution_manifest": BASELINE_OUTPUT_ROOT / "baseline_execution_manifest.json",
    "drive_archive": DRIVE_ROOT / "archives" / "external_baseline_outputs" / f"{RUN_ID}.zip",
}
for name, path in paths.items():
    print(name, "=", path, "exists=", path.exists())
